# Migration: audit columns on silver_person_alias (one-off, 2026-10-17)

Asana 1218716867008994. Adds `created_at`, `created_by`, `updated_at`, `updated_by` to the hand-maintained `silver_person_alias`. `created_at` and `created_by` default on INSERT (so an MCP `execute_write_sql` INSERT that omits them records who and when); set `updated_at` / `updated_by` on UPDATE. Existing rows keep NULL: their origin is unknown and is not back-filled.

Run once, before `notebook_01_document_matching` on the branch that expects the columns. Safe to re-run: columns already present are skipped and the defaults are re-set. Rows are never touched. Done in a migration, not in notebook_01, because `ADD COLUMNS IF NOT EXISTS` is not supported on this runtime and `CREATE TABLE IF NOT EXISTS` never alters an existing table.

In [0]:
%sql
-- Informational: audit columns present before the migration (expect none) and after (expect four).
SELECT column_name FROM workspace.information_schema.columns
WHERE table_schema = 'genealogy' AND table_name = 'silver_person_alias'
  AND column_name IN ('created_at', 'created_by', 'updated_at', 'updated_by');

In [0]:
T = "genealogy.silver_person_alias"
have = {r["col_name"] for r in spark.sql(f"DESCRIBE TABLE {T}").collect()}
spark.sql(f"ALTER TABLE {T} SET TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')")
for col, typ in [("created_at", "TIMESTAMP"), ("created_by", "STRING"), ("updated_at", "TIMESTAMP"), ("updated_by", "STRING")]:
    if col not in have:
        spark.sql(f"ALTER TABLE {T} ADD COLUMNS ({col} {typ})")
spark.sql(f"ALTER TABLE {T} ALTER COLUMN created_at SET DEFAULT current_timestamp()")
spark.sql(f"ALTER TABLE {T} ALTER COLUMN created_by SET DEFAULT current_user()")

In [0]:
%sql
-- Verify: four audit columns, row count unchanged, existing rows NULL (expect existing_rows_with_audit = 0)
DESCRIBE TABLE genealogy.silver_person_alias;

In [0]:
%sql
SELECT COUNT(*) AS alias_rows, COUNT(created_at) AS existing_rows_with_audit
FROM genealogy.silver_person_alias;